# YOLOv8 Arrow Detection Training (COCO Export)

This notebook converts the COCO export to YOLO format, creates a train/val split, then trains a YOLOv8 model.

In [4]:
# Optional (run if dependencies are not installed yet)
# !pip install -r requirements.txt

In [1]:
from pathlib import Path
import json
import random
import shutil
import yaml
from ultralytics import YOLO

In [2]:
# Paths + split config
COCO_JSON = Path('game arrows.coco/train/_annotations.coco.json')
IMAGE_DIR = COCO_JSON.parent
OUTPUT_ROOT = Path('arrow_dataset')
TRAIN_RATIO = 0.8
SEED = 42

assert COCO_JSON.exists(), f'Missing COCO JSON: {COCO_JSON.resolve()}'
print('COCO JSON:', COCO_JSON.resolve())

COCO JSON: /home/xd/Documents/py_codes/arrows/game arrows.coco/train/_annotations.coco.json


In [3]:
# Convert COCO -> YOLO + split into train/val
with COCO_JSON.open('r', encoding='utf-8') as f:
    coco = json.load(f)

# We want class order: left, right, up, down (direction-sensitive)
name_order = ['left', 'right', 'up', 'down']
name_to_index = {name: idx for idx, name in enumerate(name_order)}
id_to_name = {c['id']: c['name'] for c in coco['categories']}
id_to_index = {cid: name_to_index[name] for cid, name in id_to_name.items() if name in name_to_index}

images = coco['images']
random.seed(SEED)
random.shuffle(images)
split_idx = int(len(images) * TRAIN_RATIO)
train_images = images[:split_idx]
val_images = images[split_idx:]

annotations_by_image = {}
for ann in coco['annotations']:
    annotations_by_image.setdefault(ann['image_id'], []).append(ann)

def prepare_split(split_name, split_images):
    img_out = OUTPUT_ROOT / 'images' / split_name
    lbl_out = OUTPUT_ROOT / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img in split_images:
        src = IMAGE_DIR / img['file_name']
        dst = img_out / img['file_name']
        if not dst.exists():
            shutil.copy2(src, dst)

        label_lines = []
        for ann in annotations_by_image.get(img['id'], []):
            if ann['category_id'] not in id_to_index:
                continue
            class_id = id_to_index[ann['category_id']]
            x, y, w, h = map(float, ann['bbox'])
            x_center = (x + w / 2) / img['width']
            y_center = (y + h / 2) / img['height']
            w_norm = w / img['width']
            h_norm = h / img['height']
            label_lines.append('{:d} {:.6f} {:.6f} {:.6f} {:.6f}'.format(class_id, x_center, y_center, w_norm, h_norm))

        label_path = lbl_out / f'{Path(img["file_name"]).stem}.txt'
        label_path.write_text('\n'.join(label_lines))

prepare_split('train', train_images)
prepare_split('val', val_images)

DATA_YAML = OUTPUT_ROOT / 'data.yaml'
data_yaml = {
    'path': str(OUTPUT_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': name_order,
    'nc': len(name_order)
}
DATA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False))

print('Created:', DATA_YAML.resolve())
print('Train images:', len(train_images), 'Val images:', len(val_images))

Created: /home/xd/Documents/py_codes/arrows/arrow_dataset/data.yaml
Train images: 44 Val images: 12


In [4]:
# Training settings tuned for small, direction-sensitive dataset
MODEL_WEIGHTS = 'yolov8n.pt'
EPOCHS = 120
IMGSZ = 640
BATCH = 8
DEVICE = 0  # set to 'cpu' if no GPU
PROJECT = 'runs'
RUN_NAME = 'arrow_yolov8n_coco_small'

In [5]:
# Train YOLOv8 (augmentations avoid flips to preserve direction labels)
model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME,
    patience=30,
    lr0=0.002,
    lrf=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    mosaic=0.8,
    mixup=0.1,
    fliplr=0.0,
    flipud=0.0
)
train_results

Ultralytics 8.4.19 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11887MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=arrow_dataset/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=arrow_yolov8n_coco_small, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x734c9371a600>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [8]:
# Validate using best checkpoint
best_model_path = "/home/xd/Documents/open_src/ComfyUI/runs/detect/runs/arrow_yolov8n_coco_small/weights/best.pt"
best_model = YOLO(best_model_path)
val_metrics = best_model.val(data=str(DATA_YAML))
val_metrics

Ultralytics 8.4.19 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11887MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7186.0±1397.5 MB/s, size: 199.0 KB)
val: Scanning /home/xd/Documents/py_codes/arrows/arrow_dataset/labels/val.cache... 12 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12/12 5.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.9it/s 0.2s
                   all         12         52      0.946      0.996      0.995      0.597
                  left         11         14          1      0.982      0.995      0.582
                 right         11         12      0.944          1      0.995      0.554
                    up          8          8       0.84          1      0.995      0.638
                  down         10         18          1          1      0.995      0.615
Speed: 0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x734b8e1a7d40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [9]:
# Run predictions on validation images
pred_results = best_model.predict(
    source=str(OUTPUT_ROOT / 'images' / 'val'),
    conf=0.25,
    save=True,
    project=PROJECT,
    name=f'{RUN_NAME}_predict'
)
print('Saved predictions to:', Path(PROJECT) / f'{RUN_NAME}_predict')


image 1/12 /home/xd/Documents/py_codes/arrows/arrow_dataset/images/val/arrow_20260301_073931_981117_jpg.rf.bokdKDCGFOseMcXmYiCQ.jpg: 384x640 1 left, 1 right, 1 up, 1 down, 36.2ms
image 2/12 /home/xd/Documents/py_codes/arrows/arrow_dataset/images/val/arrow_20260301_073935_198825_jpg.rf.Q6WBOupVeHXYRosj9SyC.jpg: 384x640 1 left, 1 right, 1 up, 1 down, 4.6ms
image 3/12 /home/xd/Documents/py_codes/arrows/arrow_dataset/images/val/arrow_20260301_073936_206393_jpg.rf.BBO4VSXrxo1kSrjPinDm.jpg: 384x640 1 left, 1 right, 1 up, 1 down, 4.2ms
image 4/12 /home/xd/Documents/py_codes/arrows/arrow_dataset/images/val/arrow_20260301_073945_662590_jpg.rf.rSKl3GUMj1Dt1cqwZPry.jpg: 384x640 1 right, 1 up, 1 down, 4.3ms
image 5/12 /home/xd/Documents/py_codes/arrows/arrow_dataset/images/val/arrow_20260301_073949_888251_jpg.rf.Im5WpO8TupYd4tNpK7uq.jpg: 384x640 1 left, 1 right, 1 up, 1 down, 4.1ms
image 6/12 /home/xd/Documents/py_codes/arrows/arrow_dataset/images/val/arrow_20260301_075342_269818_jpg.rf.D1KsCec1K

## Notes
- Best weights are saved at: `runs/arrow_yolov8n_coco_small/weights/best.pt`.
- Augmentations disable flips to preserve left/right/up/down semantics.
- If training overfits quickly, reduce `EPOCHS` or increase `patience`/augmentations.